# Latest Code

## Importing Libraries and Loading env

In [ ]:
import re
import json
import os
import time

from dotenv import load_dotenv
from datasets import Dataset

from rag_eval_adapter import query_rag

from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    answer_similarity,
    answer_correctness,
    context_precision,
    context_recall,
)

from ragas.llms import llm_factory

from openai import OpenAI
from langchain_core.messages import AIMessage
from langchain_ollama import ChatOllama
from langchain_ollama import OllamaEmbeddings
from sentence_transformers import SentenceTransformer


load_dotenv()

C:\Users\Nakul\AppData\Local\Temp\ipykernel_3204\1136676870.py:13: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\Nakul\AppData\Local\Temp\ipykernel_3204\1136676870.py:13: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\Nakul\AppData\Local\Temp\ipykernel_3204\1136676870.py:13: DeprecationWarning: Importing answer_similarity from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_similarity
  from ragas.metrics import (
C:\Users\Nakul\AppData\Local\Temp\ipykernel_32

True

## Hugging Face Authentication

In [ ]:
# Hugging Face Authentication
hf_token = os.getenv("HF_TOKEN")

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token
    print("✅ Hugging Face Token Loaded")
else:
    print("⚠️ HF_TOKEN not found")

✅ Hugging Face Token Loaded


## Configure environment variables

In [ ]:
BENCHMARK_FILE = "rag_benchmark_50q.json"
CACHE_FILE = "rag_outputs.json"
RESULTS_FILE = "evaluation_results.json"
GEN_TIME_FILE = "generation_times.json"
JUDGE_MODEL ="nvidia/nemotron-3-ultra-550b-a55b:free"

## Ollama Judge

In [ ]:
class JSONSafeChatOllama(ChatOllama):

    def _extract_and_fix_json(self, text: str) -> str:

        # 1. Already valid JSON — do nothing
        try:
            json.loads(text)
            return text
        except json.JSONDecodeError:
            pass

        # 2. Extract JSON block from surrounding plain text
        for pattern in [r'(\{.*\})', r'(\[.*\])']:
            match = re.search(pattern, text, re.DOTALL)
            if match:
                candidate = match.group(1)
                # Fix unescaped backslashes (LaTeX, etc.)
                fixed = re.sub(r'\\(?!["\\/bfnrtu])', r'\\\\', candidate)
                try:
                    json.loads(fixed)
                    return fixed
                except json.JSONDecodeError:
                    pass

        # 3. Fix backslashes on raw text as last resort
        fixed = re.sub(r'\\(?!["\\/bfnrtu])', r'\\\\', text)
        try:
            json.loads(fixed)
            return fixed
        except json.JSONDecodeError:
            return text  # give up, let RAGAS handle the error

    def invoke(self, input, config=None, **kwargs):
        result = super().invoke(input, config, **kwargs)
        if isinstance(result.content, str):
            result.content = self._extract_and_fix_json(result.content)
        return result

    async def ainvoke(self, input, config=None, **kwargs):
        result = await super().ainvoke(input, config, **kwargs)
        if isinstance(result.content, str):
            result.content = self._extract_and_fix_json(result.content)
        return result
    

judge_llm = JSONSafeChatOllama(
    model="qwen2.5:7b",
    temperature=0,
    num_ctx=8192,
)

## Embedding Model(Ollama)

In [ ]:
OllamaEmbeddings(
    model="nomic-embed-text-v2-moe:latest"
)

OllamaEmbeddings(model='nomic-embed-text-v2-moe:latest', base_url=None, client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

## Embedding Model(Hugging Face)

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from ragas.embeddings import LangchainEmbeddingsWrapper

hf_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

embedding_model = LangchainEmbeddingsWrapper(hf_embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5245.68it/s]
C:\Users\Nakul\AppData\Local\Temp\ipykernel_3204\1733445127.py:8: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  embedding_model = LangchainEmbeddingsWrapper(hf_embeddings)


## Build / Load Dataset

In [ ]:
if os.path.exists(CACHE_FILE):

    print(f"Loading cached outputs from {CACHE_FILE}")

    with open(CACHE_FILE, "r", encoding="utf-8") as f:
        evaluation_rows = json.load(f)

    if os.path.exists(GEN_TIME_FILE):
        with open(GEN_TIME_FILE, "r", encoding="utf-8") as f:
            generation_times = json.load(f)

        print(
            f"Loaded {len(generation_times)} cached generation times"
        )

else:

    print("Generating RAG outputs...")

    evaluation_rows = []
    generation_times = []
    with open(BENCHMARK_FILE, "r", encoding="utf-8") as f:
        benchmark = json.load(f)

    for idx, sample in enumerate(benchmark, start=1):

        question = sample["question"]

        print(f"\n========== QUESTION {idx} ==========")
        print(question)

        print(f"[{idx}/{len(benchmark)}] Processing")

        start = time.time()

        rag_answer, retrieved_contexts, _ = query_rag(question,5)

        generation_time = time.time() - start
        generation_times.append(generation_time)

        print(
            f"QUESTION {idx} COMPLETED "
            f"({generation_time:.2f} sec)"
        )

        evaluation_rows.append(
            {
                "user_input": question,
                "response": rag_answer,
                "retrieved_contexts": retrieved_contexts,
                "reference": sample["reference_answer"],
            }
        )

        # Save after every question
        with open(CACHE_FILE, "w", encoding="utf-8") as f:
            json.dump(
                evaluation_rows,
                f,
                indent=2,
                ensure_ascii=False,
            )

        with open(GEN_TIME_FILE, "w", encoding="utf-8") as f:
            json.dump(
                generation_times,
                f,
                indent=2,
            )

    print(f"Saved outputs to {CACHE_FILE}")
    print(f"Saved generation times to {GEN_TIME_FILE}")

Loading cached outputs from rag_outputs.json
Loaded 50 cached generation times


## Dataset

In [ ]:
eval_dataset = Dataset.from_list(evaluation_rows)

print(f"Dataset Size: {len(eval_dataset)}")

Dataset Size: 50


## Metric Configuration

In [ ]:
faithfulness.llm = ragas_llm
context_precision.llm = ragas_llm
context_recall.llm = ragas_llm

answer_relevancy.llm = ragas_llm
answer_relevancy.embeddings = embedding_model

answer_similarity.embeddings = embedding_model

answer_correctness.llm = ragas_llm
answer_correctness.embeddings = embedding_model

metrics = [
    faithfulness,
    answer_relevancy,
    answer_similarity,
    answer_correctness,
    context_precision,
    context_recall,
]

## Evaluation

In [ ]:
print("\nStarting Evaluation...\n")

eval_start = time.time()

results = evaluate(
    dataset=eval_dataset,
    metrics=metrics,
    batch_size=1,
)

eval_time = time.time() - eval_start


Starting Evaluation...



Evaluating:  70%|███████   | 211/300 [54:06<30:34, 20.61s/it]  Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt response_relevance_prompt failed to parse output: The output parser failed to parse the output including retries.
Exception raised in Job[211]: RagasOutputParserException(The output parser failed to parse the output including retries.)
Evaluating:  71%|███████   | 213/300 [54:20<19:32, 13.47s/it]Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser f

## Results

In [ ]:
results_dict = results.to_pandas().mean(numeric_only=True).to_dict()

if "faithfulness" in results_dict:
    results_dict["hallucination_rate"] = (
        1 - results_dict["faithfulness"]
    )

if generation_times:
    results_dict["avg_generation_latency_sec"] = (
        sum(generation_times) / len(generation_times)
    )
    results_dict["total_generation_time_sec"] = (
    sum(generation_times)
    )

results_dict["evaluation_time_sec"] = eval_time

print("\n===== RESULTS =====")

for k, v in results_dict.items():
    print(f"{k}: {v}")
    



===== RESULTS =====
faithfulness: 0.8971190476190476
answer_relevancy: 0.7627602784789018
answer_similarity: 0.8125515546280618
answer_correctness: 0.7332914319831145
context_precision: 0.8525226756753133
context_recall: 0.8187777777777778
hallucination_rate: 0.1028809523809524
avg_generation_latency_sec: 25.05670147895813
total_generation_time_sec: 1252.8350739479065
evaluation_time_sec: 5293.184508800507


## Save Results

In [ ]:
with open(RESULTS_FILE, "w", encoding="utf-8") as f:
    json.dump(
        results_dict,
        f,
        indent=2,
        ensure_ascii=False,
    )

print(f"\nSaved results to {RESULTS_FILE}")


Saved results to evaluation_results.json
